# GNN-BERT Music Context Demo

This notebook runs inference on a real processed FMA track using:
- the leakage-safe FMA test manifest
- a saved real-data GNN-BERT Task 3 checkpoint
- the stored PyTorch Geometric audio graph
- the real FMA metadata text context and genre targets

The DEAM audio/valence-arousal manifests are available separately under `data/splits/deam/`; this demo keeps the prediction labels consistent with the FMA checkpoint.

In [ ]:
import json
import os
import sys
# A token is optional. Never place a real token in this notebook.
if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is available; authenticated Hub access is enabled.")
else:
    print("No HF_TOKEN found. This is safe; cached model files can still be used.")
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import torch

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src").is_dir():
        ROOT = candidate
        break
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from fma_dataset import load_fma_label_names
from train import FusionTrainingModel

MANIFEST_ROOT = ROOT / "data" / "splits"
CHECKPOINT = ROOT / "checkpoints" / "expanded_fma" / "task3_epoch_1.pt"
print("Project root:", ROOT)
print("Checkpoint exists:", CHECKPOINT.exists())
print("Torch version:", torch.__version__)


In [ ]:
labels = load_fma_label_names(MANIFEST_ROOT)
manifest_path = MANIFEST_ROOT / "test.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"FMA test manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if not manifest:
    raise ValueError(f"FMA test manifest is empty: {manifest_path}")
record = manifest[0]
labels = load_fma_label_names(MANIFEST_ROOT)
graph_path = Path(record["graph_path"])
if not graph_path.is_absolute():
    graph_path = ROOT / graph_path
if not graph_path.is_file():
    raise FileNotFoundError(f"Stored graph not found for track {record.get('track_id')}: {graph_path}")
graph = torch.load(graph_path, map_location="cpu", weights_only=False)
text_context = record["text_context"]
reference_tags = record.get("tags", [])

print("Track ID:", record["track_id"])
print("Graph:", graph_path)
print("Text context:", text_context)
print("Reference tags:", reference_tags)
print("Graph nodes:", graph.num_nodes)
print("Graph edges:", graph.edge_index.size(1))

In [ ]:
# Load the real trained Task 3 model and run graph/text fusion inference.
# Any HF warning is about download limits only; it is not a model failure.
model = FusionTrainingModel(task="task3", num_tags=len(labels))
model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu"))
model.eval()

with torch.no_grad():
    graph_embedding, hidden_text = model.encode(graph, [text_context])
    tag_logits, valence_pred, arousal_pred = model.head(graph_embedding, hidden_text)

print("Text representation shape:", tuple(hidden_text.shape))
print("Graph representation shape:", tuple(graph_embedding.shape))
print("Tag logits shape:", tuple(tag_logits.shape))

In [ ]:
# Decode predictions and compare them with the held-out record labels
probabilities = torch.sigmoid(tag_logits[0])
top_indices = torch.argsort(probabilities, descending=True)[:10].tolist()
predicted_tags = [labels[index] for index in top_indices if probabilities[index] >= 0.5]

print("Predicted tags:", predicted_tags or [labels[index] for index in top_indices[:5]])
print("Reference FMA tags:", reference_tags)
print(f"Predicted valence: {valence_pred[0, 0].item():.4f}")
print(f"Predicted arousal: {arousal_pred[0, 0].item():.4f}")
print("Note: this FMA checkpoint has no DEAM regression targets; use data/splits/deam for verified emotion labels.")

In [ ]:
# Inspect the real graph structure
G = nx.Graph()
G.add_nodes_from(range(graph.num_nodes))
for source, target in graph.edge_index.t().tolist():
    if source != target:
        G.add_edge(source, target)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color="steelblue", node_size=700, edge_color="gray")
plt.title(f"FMA track {record['track_id']} segment graph")
plt.tight_layout()
plt.show()

In [ ]:
# Show the verified DEAM dataset is available for the emotion extension
DEAM_REPORT = ROOT / "data" / "splits" / "deam" / "alignment_report.json"
print(json.loads(DEAM_REPORT.read_text(encoding="utf-8")))

## Summary

This demo uses real FMA graph data, real metadata text, and a saved real-data GNN-BERT checkpoint. DEAM audio and emotion targets are also downloaded and exactly paired in their own manifests, but are not mixed with FMA because their dataset IDs are unrelated.